# 04 — Detector noise, cube reconstruction, SNR

**Portfolio / learning project — not a calibrated instrument simulator.** All quantities below are synthetic proxies, literature-derived presets, or engineering estimates (see the README provenance table).

This notebook is narrative and self-contained; run the cells top to bottom.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
FIG = pathlib.Path.cwd().parent / 'figures' / 'generated'
FIG.mkdir(parents=True, exist_ok=True)


## Add noise, then invert
We expose the detector frame (Poisson + read + dark + flat + bad pixels), then jointly deblend overlapping traces with inverse-variance least squares. The same extraction returns its matching variance cube for the SNR map; nearest-pixel variance must not be paired with joint-extraction flux.

In [ ]:
from ao_fed_ifu.source_cube import SourceCubeConfig, create_emission_line_cube, scale_cube_to_total_electrons
from ao_fed_ifu.spectrograph_model import SpectrographConfig
from ao_fed_ifu.ifu_mapping import IFUConfig, build_slicer_mapping, forward_project_cube
from ao_fed_ifu.detector_model import DetectorConfig, calibrate_readout, expose
from ao_fed_ifu import cube_reconstruction as recon
from ao_fed_ifu import diagnostics as diag

spectro = SpectrographConfig(lambda_min_um=2.0, lambda_max_um=2.3, resolving_power_R=2500.0)
cfg = SourceCubeConfig(nx=12, ny=12, nlambda=spectro.n_detector_pixels, lambda_min_um=2.0, lambda_max_um=2.3)
wl = cfg.wavelength_grid_um; centre = spectro.lambda_mid_um
cube = create_emission_line_cube(cfg, line_center_um=centre, line_flux_rel=12.0,
        line_sigma_um=0.0003, velocity_gradient_kms_per_spaxel=15.0)
cube = scale_cube_to_total_electrons(cube, 3e6)
ifu = IFUConfig(nx_spaxel=cfg.nx, ny_spaxel=cfg.ny, cross_talk_sigma_pix=0.6)
mapping = build_slicer_mapping(ifu, spectro, wl)
frame = forward_project_cube(cube, mapping)
det = DetectorConfig(read_noise_e=10.0)
readout = expose(frame, det, exposure_s=600.0, rng=0)
# Remove the known bias/dark expectation and pixel response (as the pipeline
# does) before extraction; bad and saturated pixels stay masked.
calibrated = calibrate_readout(readout, det, exposure_s=600.0)
rebuilt, var_cube = recon.reconstruct_cube_weighted_with_variance(
    calibrated.image_e, calibrated.variance_e2, mapping)
print('reconstructed cube shape:', rebuilt.shape)

In [ ]:
window = 0.0012
lf = diag.line_flux_map(rebuilt, wl, centre, window, variance_cube=var_cube)  # same mask as SNR
snr = diag.snr_map(rebuilt, var_cube, wl, centre, window)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
im0 = ax[0].imshow(lf, origin='lower', cmap='viridis'); ax[0].set_title('line flux [e-]'); plt.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(snr, origin='lower', cmap='cividis'); ax[1].set_title('SNR'); plt.colorbar(im1, ax=ax[1])
fig.tight_layout(); fig.savefig(FIG / 'nb04_snr_map.png', dpi=110); plt.show()

## SNR scales as sqrt(exposure) in the photon limit

In [ ]:
rn = DetectorConfig(read_noise_e=3.0, dark_current_e_s_pix=0.0)
rate = 200.0
ts = np.array([10, 20, 40, 80, 160, 320.0])
snr_t = []
for t in ts:
    var = rate*t + rn.read_noise_e**2
    snr_t.append(rate*t / np.sqrt(var))
slope = np.polyfit(np.log(ts), np.log(snr_t), 1)[0]
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.loglog(ts, snr_t, 'o-'); ax.set_xlabel('exposure [s]'); ax.set_ylabel('SNR')
ax.set_title(f'photon-limited SNR, slope={slope:.2f} (expect 0.5)'); ax.grid(alpha=0.3, which='both')
fig.tight_layout(); fig.savefig(FIG / 'nb04_snr_sqrt_t.png', dpi=110); plt.show()
pd.DataFrame({'exposure_s': ts, 'snr': snr_t}).to_csv(FIG / 'nb04_summary.csv', index=False)